# 02 - Path Construction and Signature Features

Build rolling windows -> channels -> normalize -> signature feature matrices for all three horizons (micro, intraday, macro), fitting each horizon's normalization scale on the reference ("normal") period only.

In [1]:
import sys
sys.path.insert(0, '..')
import yaml
import numpy as np
import pandas as pd

from src.ingestion.yahoo import load_daily
from src.ingestion.binance import fetch_klines, fetch_trades
from src.features.pipeline import (
    HorizonConfig, fit_reference_scale, fit_reference_scale_multi,
    raw_paths_by_reference_period, build_feature_frame, raw_paths_for_horizon,
)

cfg = yaml.safe_load(open('../configs/horizons.yaml'))
periods = cfg['reference_periods']  # macro: multi-period, auto-detected calm stretches (2015-2026)
intraday_ref = cfg['intraday_reference_period']  # intraday: still single-period (Binance-only, not yet event-validated)
symbol = cfg['asset']

## Macro horizon (Yahoo daily)

In [2]:
macro_cfg = HorizonConfig.from_dict('macro', cfg['horizons']['macro'])

daily = load_daily(cfg['yahoo_ticker'])
if daily.index.tz is None:
    daily.index = daily.index.tz_localize('UTC')

macro_scale = fit_reference_scale_multi(daily, macro_cfg, periods)
macro_features = build_feature_frame(daily, macro_cfg, macro_scale)
print(macro_features.shape)
macro_features.to_parquet('../data/processed/macro_features.parquet')
macro_features.head()

(434, 780)


,sig_0,sig_1,sig_2,sig_3,sig_4,sig_5,sig_6,sig_7,sig_8,sig_9,...,sig_770,sig_771,sig_772,sig_773,sig_774,sig_775,sig_776,sig_777,sig_778,sig_779
window_end,,,,,,,,,,,,,,,,,,,,,
2014-10-16 00:00:00+00:00,1.0,-8.855721,0.002041,3.179307,-0.632565,0.5,0.119068,-0.000732,0.107049,-0.616548,...,-0.503251,6.594392,0.004105,-4.513756,7.865190,0.054963,1.967417,-0.001137,-1.942069,6.671284e-03
2014-10-26 00:00:00+00:00,1.0,-5.901540,0.000852,2.072949,0.608725,0.5,-1.966277,-0.001530,-0.625597,0.463339,...,-1.424757,19.126471,0.006857,-1.164776,4.796597,0.102101,-0.181759,-0.001443,-2.168906,5.721005e-03
2014-11-05 00:00:00+00:00,1.0,0.484359,0.001498,1.794086,0.565630,0.5,-3.106488,-0.000206,-0.322905,0.639300,...,-2.375809,20.170636,0.004689,2.003218,5.081704,0.051063,0.129967,-0.001416,-2.297422,4.264994e-03
2014-11-15 00:00:00+00:00,1.0,-0.995461,0.001189,4.239917,4.497871,0.5,2.073186,-0.000245,2.045901,4.539627,...,-2.110835,168.516343,0.097490,61.656022,38.228259,1.185517,-47.382983,0.002074,-62.664619,1.705362e+01
2014-11-25 00:00:00+00:00,1.0,3.046330,0.001924,2.467643,-0.039000,0.5,1.998676,0.000236,-0.006253,0.033797,...,-14.909182,174.050054,0.071294,127.665794,211.862393,1.099671,-39.629460,-0.001973,-70.169835,9.638845e-08


## Intraday horizon (Binance 1m klines resampled to hourly)

In [3]:
intraday_cfg = HorizonConfig.from_dict('intraday', cfg['horizons']['intraday'])

klines_1m = fetch_klines(symbol, '1m', intraday_ref['start'], intraday_ref['end'])
hourly_ref = klines_1m.resample(cfg['horizons']['intraday']['resample']).agg(
    {'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last', 'volume': 'sum'}
).dropna()

intraday_scale = fit_reference_scale(hourly_ref, intraday_cfg)
print('intraday scale:', intraday_scale)

intraday scale: [2.83971141e-16 4.37294478e-03 1.51829887e+03]


In [4]:
# For scoring beyond the reference period, resample a wider klines pull the same way.
# (Extend the date range here once you have event-window klines from notebook 01.)
intraday_features = build_feature_frame(hourly_ref, intraday_cfg, intraday_scale)
print(intraday_features.shape)
intraday_features.to_parquet('../data/processed/intraday_features_reference.parquet')
intraday_features.head()

(75, 120)


,sig_0,sig_1,sig_2,sig_3,sig_4,sig_5,sig_6,sig_7,sig_8,sig_9,...,sig_110,sig_111,sig_112,sig_113,sig_114,sig_115,sig_116,sig_117,sig_118,sig_119
window_end,,,,,,,,,,,,,,,,,,,,,
2023-10-02 23:00:00+00:00,1.0,4.226049,1.113716,0.5,-1.004200,-0.016619,5.230248,8.929744,-4.935916,1.130335,...,54.224628,0.329349,-3.951581,-1.165023,15.673819,-29.857145,-71.158929,0.752508,31.357783,0.064104
2023-10-03 23:00:00+00:00,1.0,-4.177170,0.738993,0.5,-2.765542,-0.438801,-1.411628,8.724375,6.040258,1.177794,...,-12.043840,0.543809,-3.597498,-0.848249,-5.736903,45.118430,17.436222,0.495011,-8.440276,0.012427
2023-10-04 23:00:00+00:00,1.0,2.887655,0.635004,0.5,2.325077,-0.162949,0.562578,4.169275,0.712373,0.797953,...,-0.587004,0.184379,0.987833,-0.202829,-0.204724,-1.656417,0.940589,0.149499,-0.356034,0.006775
2023-10-05 23:00:00+00:00,1.0,0.835469,0.307055,0.5,-1.392994,-0.517490,2.228463,0.349004,5.743520,0.824544,...,-28.564153,0.209661,-0.920247,-0.779891,-1.106368,21.333165,31.692867,0.313155,-11.693415,0.000370
2023-10-06 23:00:00+00:00,1.0,0.978913,0.582859,0.5,2.190748,-0.364853,-1.211835,0.479136,2.007365,0.947712,...,-11.897026,0.302651,1.426028,-1.039298,-6.889357,-14.136070,14.299746,0.484913,-5.648841,0.004809


## Micro horizon (Binance raw ticks, reference period)

In [5]:
micro_cfg = HorizonConfig.from_dict('micro', cfg['horizons']['micro'])
tick_ref = cfg['tick_reference_period']

trades_ref = fetch_trades(symbol, tick_ref['start'], tick_ref['end'])
half = len(trades_ref) // 2
# fit scale on the first half, score the second half -- avoid fitting
# normalization on the exact same windows it's later applied to
micro_scale = fit_reference_scale(trades_ref.iloc[:half], micro_cfg)
micro_features = build_feature_frame(trades_ref.iloc[half:], micro_cfg, micro_scale)
print(micro_features.shape)
micro_features.to_parquet('../data/processed/micro_features_reference.parquet')
micro_features.head()

(5947, 39)


,sig_0,sig_1,sig_2,sig_3,sig_4,sig_5,sig_6,sig_7,sig_8,sig_9,...,sig_29,sig_30,sig_31,sig_32,sig_33,sig_34,sig_35,sig_36,sig_37,sig_38
window_end,,,,,,,,,,,,,,,,,,,,,
2023-10-03 13:50:20.577000+00:00,346.512248,-3.212658,-5.935195,60035.369158,-1137.717422,3194.173759,24.491930,5.160587,-61.981650,-5250.791635,...,471.703663,-7.385147e+05,-5012.887184,-134038.679960,37738.005465,-433.199749,-575.534127,82601.576951,47.245039,-34.846069
2023-10-03 13:50:45.672000+00:00,346.512248,-17.238818,28.194674,60035.369158,-3313.867446,2788.039038,-2659.594193,148.588426,30.359356,6981.760865,...,308.136680,1.018668e+06,-59629.842267,31932.594991,-89158.395373,5882.781861,239.698789,82457.938492,-7399.745465,3735.510701
2023-10-03 13:51:05.125000+00:00,346.512248,-10.605039,14.323818,60035.369158,1831.769373,4935.220472,-5506.545144,56.233422,-322.189961,28.157994,...,-3109.205027,-1.996954e+05,-6228.991714,-17510.149910,28252.939355,538.763844,1603.419618,8956.739947,417.858143,489.807188
2023-10-03 13:51:28.421000+00:00,346.512248,7.535895,15.274806,60035.369158,1351.374439,1403.893546,1259.905621,28.394860,15.641978,3889.013798,...,91.858268,5.508383e+05,15447.561918,7236.841101,11982.384692,319.240947,55.211641,26083.544936,732.066502,593.985512
2023-10-03 13:51:51.633000+00:00,346.512248,18.248525,16.464464,60035.369158,5228.004182,3402.019822,1095.333201,166.504330,80.855469,2303.118778,...,539.215078,1.444501e+05,11764.100036,-1089.001531,18068.590755,1484.572847,252.811843,19504.309403,1681.365274,743.860635


Feature frames and fitted scales are saved to `data/processed/` and reused by notebooks 03-05. Re-run this notebook with wider date ranges (event windows from notebook 01) once you're ready to score beyond the reference period.